In [ ]:
!pip install -q langgraph langchain-core langchain-community huggingface_hub

In [ ]:
from typing import TypedDict, Annotated, Sequence
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from huggingface_hub import InferenceClient
from langgraph.graph import StateGraph, END
import operator

class GraphState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], operator.add]

client = InferenceClient(token="API")

def llm_node(state: GraphState) -> GraphState:
    messages = state["messages"]
    formatted_messages = []

    for msg in messages:
        if isinstance(msg, HumanMessage):
            formatted_messages.append({"role": "user", "content": msg.content})
        elif isinstance(msg, AIMessage):
            formatted_messages.append({"role": "assistant", "content": msg.content})

    response = client.chat_completion(
        messages=formatted_messages,
        model="meta-llama/Llama-3.2-3B-Instruct",
        max_tokens=512,
        temperature=0.7
    )

    return {"messages": [AIMessage(content=response.choices[0].message.content)]}

def create_graph():
    workflow = StateGraph(GraphState)
    workflow.add_node("llm", llm_node)
    workflow.set_entry_point("llm")
    workflow.add_edge("llm", END)
    return workflow.compile()

app = create_graph()
queries = ["Explain Python decorators", "What is the derivative of x^2?", "Who discovered gravity?"]
state = {"messages": []}

for i, query in enumerate(queries, 1):
    print(f"\n[Question {i}] User: {query}")
    state["messages"].append(HumanMessage(content=query))
    result = app.invoke(state)
    state = result
    print(f"AI: {result['messages'][-1].content}")
    print("="*60)

print(f"\nTotal messages: {len(state['messages'])}")


[Question 1] User: Explain Python decorators
AI: **Python Decorators**

Decorators are a powerful feature in Python that allows you to modify or extend the behavior of a function or class without permanently changing its implementation.

**What are Decorators?**
---------------------

A decorator is a small function that takes another function as an argument and returns a new function that "wraps" the original function. The new function produced by the decorator is then called instead of the original function when it's invoked.

**Basic Syntax**
---------------

Here's the basic syntax of a decorator:
```python
def decorator_function(original_function):
    def wrapper_function(*args, **kwargs):
        # code to be executed before the original function
        result = original_function(*args, **kwargs)
        # code to be executed after the original function
        return result
    return wrapper_function

@decorator_function
def original_function():
    pass
```
In this example,